In [2]:
import pandas as pd
import numpy as np
import  matplotlib.pyplot as plt
import glob
import os
import cmath 
import json
import seaborn as sns
from collections import Counter
from scipy.stats import linregress

from scipy.io import loadmat
import h5py


## **Data Loading**

In [3]:
data = loadmat("/Users/elodiehusson/Desktop/Projet/comportement AVANT STAGE/fitting/M5_FRINT_Fit/LucasBenjamin_5a9c41f2dbdb470001eeef9c_M5_VBMC.mat")
out = data["out_vbmc"][0,0]

## **Exploring Dataset Matlab**

In [4]:
print(out.dtype.names)

('s_sen', 'g_sen', 's_rot', 'd_rot_p', 'd_rot_m', 's_fac', 'w_fac_p', 'w_fac_m', 's_int', 'l_int', 's_rep', 'p_lap', 'pnam', 'pmap', 'pavg', 'fitalgo', 'version', 'tfit', 'nsmp', 'nres', 'nobs', 'nfit', 'vp', 'elbo', 'elbo_sd', 'xnam', 'xmap', 'xavg', 'xstd', 'xcov', 'xmed', 'xqrt', 'xc95', 'xnam_all', 'xhat_all', 'options', 'output', 'tss', 'ess', 'zang_f', 'zabs_f', 'xcon_f', 'zang_u', 'zabs_u', 'xcon_u')


In [5]:
# to get the name of a parameter :
print([str(x[0]) if len(x) > 0 else "" for x in out["xnam"].squeeze()])

# to get the value of a parameter :
print(out["xavg"].squeeze())

# to get one item from the file : 
print(out["g_sen"].item())

# to get the list of items of a parameter : 
print(out["pmap"][0].item())

['g_sen', 's_rot', 'd_rot_p', 'd_rot_m', 'w_fac_p', 'w_fac_m', 's_int', 'l_int', 's_rep']
[ 0.63433461  0.38234764  0.77842035 -0.63712435  0.34348909  0.22695099
  0.24461054  0.46630471  0.35964481]
0.6343346126390547
(array([[0]], dtype=uint8), array([[0.59565821]]), array([[0.32844894]]), array([[0.75808252]]), array([[-0.64484873]]), array([[0]], dtype=uint8), array([[0.27580335]]), array([[0.04027514]]), array([[0.27218877]]), array([[0.34341657]]), array([[0.34610924]]), array([[0]], dtype=uint8))


## **Info and Dataframes**

In [9]:
tfit = out["tfit"][0].item()
print("total fitting time =", tfit)
nfit = out["nfit"][0].item()
print("number of fitted parameters =", nfit)
xnam = out["xnam"][0]
print("the name of fitted parameters are :", [str(x[0]) if len(x) > 0 else "" for x in out["xnam"].squeeze()])

total fitting time = 2943.41071775
number of fitted parameters = 9
the name of fitted parameters are : ['g_sen', 's_rot', 'd_rot_p', 'd_rot_m', 'w_fac_p', 'w_fac_m', 's_int', 'l_int', 's_rep']


In [7]:
parameters = pd.DataFrame(index= [str(x[0]) if len(x) > 0 else "" for x in out["pnam"].squeeze()], columns =  ["value"])
for column, num in zip([str(x[0]) if len(x) > 0 else "" for x in out["pnam"].squeeze()], range(12)) : 
    parameters.loc[column] = out[num].item()

parameters["mod"] = np.squeeze(out["pmap"][0].item())
parameters["average"] = np.squeeze(out["pavg"][0].item())

parameters


,value,mod,average
s_sen,0,0.000000,0.000000
g_sen,0.634335,0.595658,0.634335
s_rot,0.382348,0.328449,0.382348
d_rot_p,0.77842,0.758083,0.778420
d_rot_m,-0.637124,-0.644849,-0.637124
s_fac,0,0.000000,0.000000
w_fac_p,0.343489,0.275803,0.343489
w_fac_m,0.226951,0.040275,0.226951
s_int,0.244611,0.272189,0.244611
l_int,0.466305,0.343417,0.466305


In [8]:
fitted_parameters = pd.DataFrame(index= [str(x[0]) if len(x) > 0 else "" for x in out["xnam"].squeeze()], columns =  ["average", "mod", "std","median", "Q1", "Q3", "c95_left", "c95_right"])
for column, num in zip([str(x[0]) if len(x) > 0 else "" for x in out["xnam"].squeeze()], range(12)) : 
    fitted_parameters["average"] = out["xavg"].squeeze()
    fitted_parameters["mod"] = out["xmap"].squeeze()
    fitted_parameters["std"] = out["xstd"].squeeze()
    fitted_parameters["median"] = out["xmed"].squeeze()
    fitted_parameters["Q1"] = out["xqrt"].squeeze()[0]
    fitted_parameters["Q3"] = out["xqrt"].squeeze()[1]
    fitted_parameters["c95_left"] = out["xc95"].squeeze()[0]
    fitted_parameters["c95_right"] = out["xc95"].squeeze()[1]

fitted_parameters


,average,mod,std,median,Q1,Q3,c95_left,c95_right
g_sen,0.634335,0.595658,0.288111,0.632266,0.438887,0.826898,0.076386,1.206571
s_rot,0.382348,0.328449,0.075081,0.376084,0.322343,0.439198,0.257854,0.528732
d_rot_p,0.778420,0.758083,0.053153,0.777023,0.741428,0.814613,0.677772,0.884195
d_rot_m,-0.637124,-0.644849,0.110639,-0.639709,-0.712872,-0.564363,-0.846471,-0.412765
w_fac_p,0.343489,0.275803,0.184480,0.304563,0.214346,0.428391,0.105003,0.809350
w_fac_m,0.226951,0.040275,0.213610,0.160232,0.086707,0.292168,0.026988,0.815285
s_int,0.244611,0.272189,0.101150,0.232802,0.168771,0.306163,0.087155,0.472631
l_int,0.466305,0.343417,0.173172,0.445612,0.342224,0.568610,0.185759,0.860807
s_rep,0.359645,0.346109,0.025956,0.359257,0.341905,0.377010,0.309863,0.411568
